In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score

In [33]:
df = pd.read_csv('/content/stores_sales_forecasting.csv', encoding='latin1')[['Ship Mode',
       'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']]
df.head()

,Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
3,Standard Class,Consumer,United States,Los Angeles,California,90032,West,Furniture,Furnishings,Eldon Expressions Wood and Plastic Desk Access...,48.8600,7,0.00,14.1694
4,Standard Class,Consumer,United States,Los Angeles,California,90032,West,Furniture,Tables,Chromcraft Rectangular Conference Tables,1706.1840,9,0.20,85.3092


In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2121 entries, 0 to 2120
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Ship Mode     2121 non-null   object 
 1   Segment       2121 non-null   object 
 2   Country       2121 non-null   object 
 3   City          2121 non-null   object 
 4   State         2121 non-null   object 
 5   Postal Code   2121 non-null   int64  
 6   Region        2121 non-null   object 
 7   Category      2121 non-null   object 
 8   Sub-Category  2121 non-null   object 
 9   Product Name  2121 non-null   object 
 10  Sales         2121 non-null   float64
 11  Quantity      2121 non-null   int64  
 12  Discount      2121 non-null   float64
 13  Profit        2121 non-null   float64
dtypes: float64(3), int64(2), object(9)
memory usage: 232.1+ KB


In [35]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Postal Code,2121.0,55726.556341,32261.888225,1040.0000,22801.000,60505.0000,90032.0000,99301.000
Sales,2121.0,349.834887,503.179145,1.8920,47.040,182.2200,435.1680,4416.174
Quantity,2121.0,3.785007,2.251620,1.0000,2.000,3.0000,5.0000,14.000
Discount,2121.0,0.173923,0.181547,0.0000,0.000,0.2000,0.3000,0.700
Profit,2121.0,8.699327,136.049246,-1862.3124,-12.849,7.7748,33.7266,1013.127


In [36]:
df.duplicated().sum()

df.drop_duplicates(inplace=True)

In [37]:
df.nunique()


,0
Ship Mode,4
Segment,3
Country,1
City,371
State,48
Postal Code,454
Region,4
Category,1
Sub-Category,4
Product Name,380


In [38]:
df = df.drop(columns=['Category','Country'])

In [39]:
num_features = ['Sales', 'Quantity', 'Discount', 'Profit']

df_cleaned = df.copy()


for feature in num_features:
    Q1 = df_cleaned[feature].quantile(0.25)
    Q3 = df_cleaned[feature].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df_cleaned = df_cleaned[(df_cleaned[feature] >= lower_bound) & (df_cleaned[feature] <= upper_bound)]

print(f"Data shape before outlier removal: {df.shape}")
print(f"Data shape after outlier removal: {df_cleaned.shape}")
df = df_cleaned.copy()

Data shape before outlier removal: (2118, 12)
Data shape after outlier removal: (1596, 12)


In [40]:
cat_features = ['Ship Mode', 'Segment', 'City', 'State', 'Postal Code', 'Region', 'Sub-Category', 'Product Name']

In [41]:
x = df.iloc[:,:11]
y = df.iloc[:,11]
x,y


(           Ship Mode      Segment          City         State  Postal Code  \
 0       Second Class     Consumer     Henderson      Kentucky        42420   
 3     Standard Class     Consumer   Los Angeles    California        90032   
 5       Second Class     Consumer  Philadelphia  Pennsylvania        19140   
 8     Standard Class     Consumer  Philadelphia  Pennsylvania        19140   
 10    Standard Class  Home Office       Houston         Texas        77041   
 ...              ...          ...           ...           ...          ...   
 2116     First Class  Home Office       Houston         Texas        77041   
 2117    Second Class    Corporate        Newark      Delaware        19711   
 2118    Second Class     Consumer     Lafayette     Louisiana        70506   
 2119    Second Class     Consumer         Miami       Florida        33180   
 2120  Standard Class     Consumer    Costa Mesa    California        92627   
 
        Region Sub-Category                       

In [42]:

x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42)

In [43]:
all_x_train_cols = x_train.columns.tolist()

current_num_features = [f for f in ['Sales', 'Quantity', 'Discount'] if f in all_x_train_cols]

current_cat_features = [f for f in cat_features if f in all_x_train_cols]

preprocessor = ColumnTransformer(
    transformers = [
        ('num', StandardScaler(), current_num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), current_cat_features)
    ]
)


In [44]:

# 1. Define the models
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "DecisionTreeRegressor": DecisionTreeRegressor(),
    "SVR": SVR(),
    "KNeighborsRegressor": KNeighborsRegressor()
}


results = {}


for name, regressor in models.items():

    current_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', regressor)
    ])


    current_pipeline.fit(x_train, y_train)


    score = current_pipeline.score(x_test, y_test)
    results[name] = score
    print(f"{name} R2 Score: {score:.4f}")

# 2. Identify the best performing model
best_model = max(results, key=results.get)
print(f"\nBest Model: {best_model} with R2: {results[best_model]:.4f}")


Linear Regression R2 Score: 0.1945
Ridge Regression R2 Score: 0.4292
Random Forest R2 Score: 0.7575
DecisionTreeRegressor R2 Score: 0.5933
SVR R2 Score: 0.2949
KNeighborsRegressor R2 Score: 0.4694

Best Model: Random Forest with R2: 0.7575
